# Create and Fill Bank Monitoring Database

This notebook renders the bank monitoring DDL with Russian comments, creates the SQLite database through the project pipeline, and previews the first rows from each generated table.

In [13]:
from contextlib import redirect_stdout
import io
import os
from pathlib import Path
import sqlite3
import sys
from IPython.display import Markdown, display
import pandas as pd

from src.config import INTERIM_DATA_DIR, PROJECT_ROOT, RAW_DATA_DIR
from src.dataset import (  # noqa: E402
    build_database,
    get_schema_description_path,
    render_table_ddls
)


COMMENT_STYLE = "inline" # "inline" or "yaml"
COMMENT_VARIANT = "short" # "short" or "business" or "technical"
PREVIEW_ROWS = 5
DATABASE_NAME = "sakila" # "bank_transaction_monitoring" or "sakila"
SCHEMA_MAPPING_PATH = os.path.join(RAW_DATA_DIR, DATABASE_NAME, "schema_mapping.json")

In [ ]:
table_ddls = render_table_ddls(
    database_name=DATABASE_NAME,
    comment_style="inline",
    comment_variant="short",
)

print(table_ddls[0])

# save as example for inference
df = pd.DataFrame({"ddl": table_ddls})
df.to_csv(f"{INTERIM_DATA_DIR}/{DATABASE_NAME}/table_ddls_inline_short.csv", index=False)

CREATE TABLE act ( -- Актёры фильмов.
  a01 numeric NOT NULL, -- Идентификатор актёра.
  a02 VARCHAR(45) NOT NULL, -- Имя актёра.
  a03 VARCHAR(45) NOT NULL, -- Фамилия актёра.
  a04 TIMESTAMP NOT NULL, -- Дата изменения записи.
  PRIMARY KEY (a01)
);


In [12]:
db_path = build_database(
    comment_style=COMMENT_STYLE,
    comment_variant=COMMENT_VARIANT,
    database_name=DATABASE_NAME,
    overwrite=True,
)

schema_description_path = get_schema_description_path(
    database_name=DATABASE_NAME,
    comment_style=COMMENT_STYLE,
    comment_variant=COMMENT_VARIANT,
)

print(f"SQLite database created: {db_path}")
print(f"Schema description: {schema_description_path}")

schema_mapping = pd.read_json(SCHEMA_MAPPING_PATH, orient="index")
table_names = schema_mapping.index.tolist()

with sqlite3.connect(db_path) as connection:
    for table_name in table_names:
        row_count = connection.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
        display(Markdown(f"## `{table_name}` ({row_count} rows)"))
        display(pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT {PREVIEW_ROWS}", connection))

SQLite database created: /home/user/cursor_projects/vanna-sql/data/processed/sakila/sakila_inline_short.sqlite.db
Schema description: /home/user/cursor_projects/vanna-sql/data/interim/sakila/schema_descriptions/schema_description_inline_short.txt


## `act` (200 rows)

,a01,a02,a03,a04
0,1,PENELOPE,GUINESS,2026-06-06 20:40:12
1,2,NICK,WAHLBERG,2026-06-06 20:40:12
2,3,ED,CHASE,2026-06-06 20:40:12
3,4,JENNIFER,DAVIS,2026-06-06 20:40:12
4,5,JOHNNY,LOLLOBRIGIDA,2026-06-06 20:40:12


## `cnt` (109 rows)

,c01,c02,c03
0,1,Afghanistan,2026-06-06 20:40:12
1,2,Algeria,2026-06-06 20:40:12
2,3,American Samoa,2026-06-06 20:40:12
3,4,Angola,2026-06-06 20:40:12
4,5,Anguilla,2026-06-06 20:40:12


...

,j01,j02,j03,j04
0,1,1,1,2026-06-06 20:40:12
1,2,2,2,2026-06-06 20:40:12


## `pay` (16049 rows)

,p01,p02,p03,p04,p05,p06,p07
0,1,1,1,76,2.99,2005-05-25 11:30:37.000,2026-06-06 20:40:12
1,2,1,1,573,0.99,2005-05-28 10:35:23.000,2026-06-06 20:40:12
2,3,1,1,1185,5.99,2005-06-15 00:54:12.000,2026-06-06 20:40:12
3,4,1,2,1422,0.99,2005-06-15 18:02:53.000,2026-06-06 20:40:12
4,5,1,2,1476,9.99,2005-06-15 21:08:46.000,2026-06-06 20:40:12


## `ren` (16044 rows)

,q01,q02,q03,q04,q05,q06,q07
0,1,2005-05-24 22:53:30.000,367,130,2005-05-26 22:04:30.000,1,2026-06-06 20:40:13
1,2,2005-05-24 22:54:33.000,1525,459,2005-05-28 19:40:33.000,1,2026-06-06 20:40:13
2,3,2005-05-24 23:03:39.000,1711,408,2005-06-01 22:12:39.000,1,2026-06-06 20:40:13
3,4,2005-05-24 23:04:41.000,2452,333,2005-06-03 01:43:41.000,2,2026-06-06 20:40:13
4,5,2005-05-24 23:05:21.000,2079,222,2005-06-02 04:33:21.000,1,2026-06-06 20:40:13
